Retrival Augumented Generation(RAG).

PDF-> Document loader -> Text Chuncking -> Embedding -> Vector DataBase ->Retrival -> LLM -> Answer Generate

In [1]:
!pip install chromadb sentence-transformers pypdf transformers

Scenario: AI Research Assistant for a Corporate Innovation Team

Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions about the papers instead of reading them cover to cover.

How the RAG Chatbot Fits In
- Input Source: The team uploads a research paper (e.g., ai_research.pdf).
- Chunking: The chatbot splits the paper into manageable sections so no detail is lost.
- Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, making the paper searchable by meaning rather than keywords.
- Retriever: When someone asks, “What does this paper say about reinforcement learning?”, the retriever pulls the most relevant chunks.
- LLM Response: The Hugging Face model (Flan-T5) generates a concise, human-readable answer using those chunks as context.
- Chat Loop: The team can keep asking questions interactively, like a research assistant that knows the paper inside out.

In [1]:
import os
from pypdf import PdfReader #library to read PDF files
from sentence_transformers import SentenceTransformer #embedding model
from transformers import pipeline # hugging face model pipeline
import chromadb #vector database

Step 2: Load the document

In [2]:
print('Loading the document....')
reader=PdfReader('/content/ai_research.pdf')
text=''
for page in reader.pages:
  text+= page.extract_text()

print('Document Loaded')
print(text[:1000])


Loading the document....
Document Loaded
Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a subset of artificial intelligence that focuses on algorithms that improve
automatically through experience. Instead of being explicitly programmed, machine learning models
learn patterns from data. Common types of machine learning include supervised learning,
unsupervised learning, and reinforcement learning.
Deep Learning
Deep learning is a specialized branch of machine learning that uses neural networks with multiple
layers to model complex patterns in dat

Step 3: Chunk the document

In [3]:
print('Splitting document into chunks..')
def chunk_text(text,chunk_size=200,overlap=20):
  chunks=[]
  start=0

  while start<len(text):
    end=start+chunk_size
    chunk=text[start:end]
    chunks.append(chunk)
    start+=chunk_size-overlap

  return chunks
chunks=chunk_text(text)
print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])

Splitting document into chunks..
Total Chunks Created: 16

Example Chunk:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, 


Step 4: Create Embeddings

In [4]:
embedding_model=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Embedding Model Loaded')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Loaded


Step 5: Create vector database

In [5]:
client=chromadb.Client()

try:
  client.delete_collection('pdf_collection')
  print('old collection deleted')
except:
  pass

collection=client.create_collection('pdf_collection')
print('New vector collection created')

New vector collection created


Step 6: Store chunks in database

In [6]:
for i, chunk in enumerate(chunks):
  embedding=embedding_model.encode(chunk).tolist()

  collection.add(documents=[chunk],
                 embeddings=[embedding],
                 ids=[str(i)])

print('All chunks are stored successfully')

All chunks are stored successfully


Step 7: Retriever Function (converts user questions -> embedding)

In [7]:
def retrieve(query,k=3):
  query_embedding=embedding_model.encode(query).tolist()

  results=collection.query(query_embeddings=[query_embedding],
                           n_results=k)

  return results['documents'][0]

In [8]:
!pip install --upgrade transformers accelerate sentencepiece

Step 8: Load the LLM

In [10]:
qa_pipeline=pipeline('text-generation',model='google/flan-t5-base')

print('LLM loaded successfully')

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully


Step 9: Question Answering Function

In [11]:
def answer_question(query):
  context_docs=retrieve(query)
  print("Retrieved docs:", context_docs)
  context=' '.join(context_docs[:2])

  prompt=f'''You are an AI assistant.
  Answer the question using ONLY the context below. If the answer is not present,
  "Not found in document".

  Context:
  {context}

  Question:
  {query}

  Answer:
  '''
  response = qa_pipeline(prompt,max_new_tokens=200,temperature=0.5,do_sample=True)
  print("RAW MODEL OUTPUT:", response)
  return response[0]['generated_text']



Step 10: Chat Loop

In [12]:
print("RAG Chatbot Ready")
print("Type 'exit' to stop")


while True:
  question = input('Ask a question: ')
  if question.lower()=='exit':
    print('Goodbye!')
    break

  answer=answer_question(question)

  print('\nAnswer:\n',answer)
  print('\n'+'-'*60+'\n')

RAG Chatbot Ready
Type 'exit' to stop
Ask a question: what is artificial intelligence


Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Retrieved docs: ['Artificial Intelligence Research Overview\nIntroduction to Artificial Intelligence\nArtificial Intelligence (AI) refers to the simulation of human intelligence in machines that are\nprogrammed to think, ', 'e healthcare\ndiagnostics, recommendation systems, autonomous vehicles, and conversational agents.\nMachine Learning\nMachine Learning is a subset of artificial intelligence that focuses on algorithms th', 'rogrammed to think, learn, and make decisions. AI systems can analyze large datasets, recognize\npatterns, and generate predictions or recommendations. Modern AI applications include healthcare\ndiagnos']


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAW MODEL OUTPUT: [{'generated_text': 'You are an AI assistant.\n  Answer the question using ONLY the context below. If the answer is not present,\n  "Not found in document".\n\n  Context:\n  Artificial Intelligence Research Overview\nIntroduction to Artificial Intelligence\nArtificial Intelligence (AI) refers to the simulation of human intelligence in machines that are\nprogrammed to think,  e healthcare\ndiagnostics, recommendation systems, autonomous vehicles, and conversational agents.\nMachine Learning\nMachine Learning is a subset of artificial intelligence that focuses on algorithms th\n\n  Question:\n  what is artificial intelligence\n\n  Answer:\n  '}]

Answer:
 You are an AI assistant.
  Answer the question using ONLY the context below. If the answer is not present,
  "Not found in document".

  Context:
  Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines th